# 02. Data Preprocessing

Prepare the processed dataset that all four team members will use.

Steps:
1. Load the raw data
2. Find the 103 signal features
3. Keep only the needed columns
4. Check features and target
5. Change infinite values to NaN
6. Inspect duplicates (report only, nothing is removed)
7. Look at how patients are grouped
8. Split by patient: 70% train / 15% validation / 15% test
9. Check the split
10. Save the split and the manifest

**Not done here:** median imputation, scaling and feature selection. They must be fitted on the training set only, so they happen later in the modelling pipeline.

## Setup

In [1]:
import sys
sys.path.append('..')

from config import config
from src.data_loader import load_dataset
from src.data_inspection import print_duplicate_summary, count_recordings_per_patient
from src.data_preprocessing import (
    get_feature_columns,
    select_columns,
    check_features_are_numeric,
    check_target_values,
    replace_inf_with_nan,
    split_by_patient,
    print_split_summary,
    check_patient_split,
    check_all_rows_assigned,
    check_both_classes,
    check_patterns_across_sets,
    save_splits,
)

## Step 1 — Load the raw data

Reads `data/raw/metadata.csv` (the raw file is never changed), renames `F_Med.1` to `F_MedAD`, and checks that `record_id`, `patient_id` and `figo_label` exist.

In [2]:
raw_df = load_dataset()

Loaded: C:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\raw\metadata.csv
Shape (rows, columns): (20769, 215)
Renamed column: F_Med.1 -> F_MedAD
Check passed: record_id, patient_id and figo_label are present.


## Step 2 — Find the 103 signal features

A feature is any column whose name starts with `F_` (FHR), `U_` (UC) or `FU_` (FHR–UC). Clinical, delivery, neonatal and `NetFea_*` columns are left out automatically. The code stops if it does not find exactly 103.

In [3]:
feature_cols = get_feature_columns(raw_df)

print("First 5 features:", feature_cols[:5])
print("Last 5 features:", feature_cols[-5:])

Number of feature columns found: 103
First 5 features: ['F_BL', 'F_A_N', 'F_D_N', 'F_ED_N', 'F_VD_N']
Last 5 features: ['FU_M_XWTTF', 'FU_V_XWTTF', 'FU_S_XWTTF', 'FU_K_XWTTF', 'FU_C_XWTTF']


## Step 3 — Keep only the needed columns

Keep `record_id`, `patient_id`, the 103 features and `figo_label`. Expected shape: (20769, 106).

In [4]:
df = select_columns(raw_df, feature_cols)
df.head()

Shape after selecting columns: (20769, 106)


,record_id,patient_id,F_BL,F_A_N,F_D_N,F_ED_N,F_VD_N,F_LD_N,F_PD_N,F_LTV,...,FU_EF_XWTTF,FU_EC_XWTTF,FU_RE_XWTTF,FU_SE_XWTTF,FU_M_XWTTF,FU_V_XWTTF,FU_S_XWTTF,FU_K_XWTTF,FU_C_XWTTF,figo_label
0,10000_1_210208000001,1,157.988333,27,5,0,5,0,0,17.968478,...,0.625249,1.700000e+14,-22.899662,-2.036245e+09,275.030959,48824.95847,0.906524,3.098757,0.803414,1
1,10003_1_210208000001,2,144.132917,8,4,2,1,1,0,16.850306,...,0.490735,1.610000e+14,-23.000586,-2.092396e+09,278.549152,58806.40840,0.754356,2.519955,0.870584,1
2,10004_1_210208000001,3,137.736667,18,0,0,0,0,0,16.757457,...,0.635695,1.670000e+14,-22.811168,-1.983650e+09,269.292660,44534.54236,0.763957,2.739259,0.783653,0
3,10005_1_210208000001,4,136.125000,12,0,0,0,0,0,16.558844,...,0.661343,1.580000e+14,-22.603509,-1.815655e+09,250.707118,35008.16836,0.750787,2.796813,0.746308,0
4,10006_1_210208000001,5,134.297500,17,0,0,0,0,0,16.540993,...,0.650610,1.760000e+14,-22.772676,-2.053460e+09,278.811377,40149.20607,0.518829,2.339740,0.718667,0


## Step 4 — Check features and target

Stops with an error if any feature column contains text, or if `figo_label` has missing values or values other than 0 and 1.

In [5]:
check_features_are_numeric(df, feature_cols)
check_target_values(df)

Check passed: all feature columns are numeric.
figo_label counts:
figo_label
1    11171
0     9598
Name: count, dtype: int64
Check passed: figo_label only contains 0 and 1.


## Step 5 — Change infinite values to NaN

The median imputer used later cannot handle `inf`, but it can fill `NaN`. This step learns nothing from the data, so it is safe to do before the split.

In [6]:
df = replace_inf_with_nan(df, feature_cols)

Infinite values changed to NaN: 0


## Step 6 — Inspect duplicates (report only)

Nothing is removed here. Several recordings from the same patient are **normal** and are not duplicates.

What we look for:
- **exact duplicate rows**: every column the same except `record_id`
- **duplicate `record_id`s**: should be 0
- **identical features, same patient, same label**: probably an accidental copy
- **identical features, different patients**: suspicious, could leak across sets
- **identical features, conflicting labels**: suspicious

Review the results and decide **as a team** before removing anything.

In [7]:
repeated_rows, pattern_summary = print_duplicate_summary(df, feature_cols)

Total records: 20769
Unique patients: 12606
Exact duplicate rows (all columns except record_id): 0
Duplicate record_id values: 0
Patients with more than one recording: 5123
Most recordings for one patient: 10

Repeated feature patterns (all features identical): 0
  - same patient, same label (likely accidental copy): 0
  - different patients, same label (suspicious): 0
  - conflicting labels (suspicious): 0
Rows involved in repeated patterns: 0


In [ ]:
# Look at the suspicious patterns (different patients or conflicting labels)
suspicious_patterns = pattern_summary[pattern_summary["type"] != "same patient"]
print("Suspicious patterns:", len(suspicious_patterns))

suspicious_rows = repeated_rows[repeated_rows["pattern_id"].isin(suspicious_patterns.index)]
suspicious_rows[["pattern_id", "record_id", "patient_id", "figo_label"]].sort_values("pattern_id")

## Step 7 — How patients are grouped

One pregnancy (`patient_id`) can have several CTG recordings made on different days. These recordings are very similar to each other.

If one recording of a patient is in **train** and another is in **test**, the model is tested on a patient it has already seen, and the scores look better than they really are. This is **data leakage**.

So we split by **patient**, not by row: all recordings of a patient go into the same set. `patient_id` is used only for grouping; it is never a feature.

The cell below shows how many recordings each patient has, and how many patients have both normal and abnormal recordings (these are kept together in one set as well).

In [8]:
# How many recordings does each patient have?
recordings_per_patient = count_recordings_per_patient(df)

print("Number of recordings -> number of patients")
print(recordings_per_patient.value_counts().sort_index())
print()
print("Average recordings per patient:", round(recordings_per_patient.mean(), 2))
print("Most recordings for one patient:", recordings_per_patient.max())

# Do some patients have both normal and abnormal recordings?
labels_per_patient = df.groupby("patient_id")["figo_label"].nunique()
mixed_label_patients = (labels_per_patient > 1).sum()
print("Patients with both normal and abnormal recordings:", mixed_label_patients)

Number of recordings -> number of patients
count
1     7483
2     3140
3     1272
4      474
5      162
6       54
7       12
8        6
9        2
10       1
Name: count, dtype: int64

Average recordings per patient: 1.65
Most recordings for one patient: 10
Patients with both normal and abnormal recordings: 3006


## Step 8 — Split by patient (70 / 15 / 15)

`StratifiedGroupKFold` cuts the data into 20 parts of about 5% each:
- every patient stays inside **one** part (grouping)
- each part keeps a similar normal/abnormal balance (stratification)

Parts 0–2 become **test** (15%), parts 3–5 become **validation** (15%), and parts 6–19 become **train** (70%). The seed is 42, so everyone gets the same split.

The percentages will be close to 70/15/15 but not exact, because patients have different numbers of recordings and a patient can't be split.

In [9]:
train_df, val_df, test_df = split_by_patient(df, seed=config.RANDOM_STATE)

c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\notebooks\..\src\data_preprocessing.py:132: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["part"] = -1


In [10]:
split_summary = print_split_summary(train_df, val_df, test_df)

   Dataset  Records  Unique Patients  Normal (0)  Abnormal (1)  Records %  Patients %  Abnormal %
  Training    14540             8742        6719          7821       70.0        69.3        53.8
Validation     3115             1904        1439          1676       15.0        15.1        53.8
      Test     3114             1960        1440          1674       15.0        15.5        53.8


## Step 9 — Check the split

Each check stops with an error if something is wrong:
- no patient appears in two sets
- every row is in exactly one set (none lost or added)
- every set contains both normal (0) and abnormal (1)

In [11]:
check_patient_split(train_df, val_df, test_df)
check_all_rows_assigned(df, train_df, val_df, test_df)
check_both_classes(train_df, val_df, test_df)

Patients in both train and validation: 0
Patients in both train and test: 0
Patients in both validation and test: 0
Check passed: every patient is in only one set.
Rows before split: 20769
Rows after split: 20769
Check passed: every row is in exactly one set.
Check passed: all three sets contain both classes.


In [12]:
# Identical feature rows in two different sets could leak information.
# This only warns; it does not change anything.
crossing_patterns = check_patterns_across_sets(train_df, val_df, test_df, feature_cols)

Check passed: no identical feature rows cross between sets.


## Step 10 — Save the split

Run this **only after** reviewing Steps 6 and 9.

Saves `train.csv`, `val.csv`, `test.csv` and `split_manifest.csv` (record_id, patient_id, split) in `data/processed/`. If any file already exists, nothing is saved.

Share `split_manifest.csv` with the team so everyone can confirm they have the same split.

In [13]:
save_splits(train_df, val_df, test_df, config.PROCESSED_DATA_DIR)

Saved: C:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\train.csv
Saved: C:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\val.csv
Saved: C:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\test.csv
Saved: C:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\data\processed\split_manifest.csv
